In [11]:
# 统计日志文件中的 QA 验证结果


import re
import os
import glob

# ================= 配置区 =================
log_file_path = 'eval_traj_20260325_004512.log'  # 请替换为你的实际 log 文件路径
traj_dir = 'eval_traj/CS_eval_traj/SFT_IterScholar/' # 轨迹文件所在的目录
# ==========================================

# 统计数据结构
stats = {
    'yes': {'count': 0, 'total_steps': 0, 'details': []},
    'no': {'count': 0, 'total_steps': 0, 'details': []}
}

# 正则提取任务 ID 和 QA 验证结果
# 匹配示例: ✅ 任务 [8] 完成！ (QA 验证: yes)
pattern = re.compile(r'任务\s*\[(\d+)\]\s*完成！.*?\(QA 验证:\s*(yes|no)\)', re.IGNORECASE)

try:
    with open(log_file_path, 'r', encoding='utf-8') as log_file:
        for line in log_file:
            match = pattern.search(line)
            if match:
                task_id = match.group(1)
                qa_result = match.group(2).lower()
                
                # 拼接并查找对应的 jsonl 轨迹文件 
                # 使用 * 忽略前面的时间戳，直接匹配 task_id
                search_pattern = os.path.join(traj_dir, f'*_CS_eval_traj_task{task_id}.jsonl')
                matched_files = glob.glob(search_pattern)
                
                steps = 0
                if matched_files:
                    # 假设每个任务只对应一个文件，取第一个匹配项
                    traj_file = matched_files[0]
                    try:
                        # 统计 JSONL 文件的行数（即步数）
                        with open(traj_file, 'r', encoding='utf-8') as tf:
                            steps = sum(1 for _ in tf)
                    except Exception as e:
                        print(f"⚠️ 无法读取轨迹文件 {traj_file}: {e}")
                else:
                    print(f"⚠️ 警告: 找不到任务 [{task_id}] 对应的轨迹文件！")
                
                # 记录统计数据
                stats[qa_result]['count'] += 1
                stats[qa_result]['total_steps'] += steps
                stats[qa_result]['details'].append((task_id, steps))

    # 输出统计报告
    print("\n📊 =============== 统计报告 =============== 📊")
    for res in ['yes', 'no']:
        count = stats[res]['count']
        total_steps = stats[res]['total_steps']
        avg_steps = (total_steps / count) if count > 0 else 0
        
        icon = "✅" if res == 'yes' else "❌"
        print(f"{icon} [QA 验证: {res.upper()}]")
        print(f"  - 任务数量: {count} 个")
        print(f"  - 消耗总步数: {total_steps} 步")
        print(f"  - 平均步数/任务: {avg_steps:.2f} 步")
        print("-" * 40)

except FileNotFoundError:
    print(f"⚠️ 找不到日志文件: {log_file_path}，请检查路径。")
except Exception as e:
    print(f"⚠️ 发生错误: {e}")


📊 =============== 统计报告 =============== 📊
✅ [QA 验证: YES]
  - 任务数量: 17 个
  - 消耗总步数: 258 步
  - 平均步数/任务: 15.18 步
----------------------------------------
❌ [QA 验证: NO]
  - 任务数量: 33 个
  - 消耗总步数: 1640 步
  - 平均步数/任务: 49.70 步
----------------------------------------


In [3]:
import os
import re
from pathlib import Path

def analyze_task_trajectories(traj_dir):
    path = Path(traj_dir)
    if not path.exists():
        print(f"错误：目录 {traj_dir} 不存在。")
        return

    results = []
    
    # 匹配文件名中包含 task 加数字的模式，例如 task1.jsonl 到 task50.jsonl
    # 考虑到文件名开头有时间戳，我们使用正则表达式提取任务 ID
    task_pattern = re.compile(r'task(\d+)\.jsonl$')

    print(f"正在扫描目录: {traj_dir} ...")

    for file_path in path.glob("*.jsonl"):
        match = task_pattern.search(file_path.name)
        if match:
            task_id = int(match.group(1))
            
            # 仅处理 1-50 范围内的任务
            if 1 <= task_id <= 50:
                try:
                    # 统计行数
                    with open(file_path, 'r', encoding='utf-8') as f:
                        line_count = sum(1 for _ in f)
                    
                    results.append({
                        "task_id": task_id,
                        "line_count": line_count,
                        "file_name": file_path.name
                    })
                except Exception as e:
                    print(f"读取文件 {file_path.name} 时出错: {e}")

    # 按照行数从小到大排序
    sorted_results = sorted(results, key=lambda x: x['line_count'])

    # 输出前 10 个
    print("\n--- 行数最少的前 10 个任务 ---")
    print(f"{'排名':<5} | {'任务 ID':<8} | {'行数':<8} | {'文件名'}")
    print("-" * 60)
    
    for i, res in enumerate(sorted_results[:10], 1):
        print(f"{i:<5} | Task {res['task_id']:<3} | {res['line_count']:<8} | {res['file_name']}")

# 执行脚本
traj_directory = 'eval_traj/CS_eval_traj/SFT_IterScholar/'
analyze_task_trajectories(traj_directory)

正在扫描目录: eval_traj/CS_eval_traj/SFT_IterScholar/ ...

--- 行数最少的前 10 个任务 ---
排名    | 任务 ID    | 行数       | 文件名
------------------------------------------------------------
1     | Task 29  | 3        | 20260325_004516_CS_eval_traj_task29.jsonl
2     | Task 19  | 5        | 20260325_004516_CS_eval_traj_task19.jsonl
3     | Task 6   | 5        | 20260325_004516_CS_eval_traj_task6.jsonl
4     | Task 1   | 6        | 20260325_004516_CS_eval_traj_task1.jsonl
5     | Task 40  | 6        | 20260325_004516_CS_eval_traj_task40.jsonl
6     | Task 47  | 6        | 20260325_004516_CS_eval_traj_task47.jsonl
7     | Task 11  | 7        | 20260325_004516_CS_eval_traj_task11.jsonl
8     | Task 43  | 7        | 20260325_004516_CS_eval_traj_task43.jsonl
9     | Task 30  | 9        | 20260325_004516_CS_eval_traj_task30.jsonl
10    | Task 5   | 9        | 20260325_004516_CS_eval_traj_task5.jsonl


In [5]:
import json
import os

def format_eval_results():
    # 1. 定义文件路径
    source_path = r'D:\DissertationProject\DissertationProject\dataset\BrowseComp-ZH\browsecomp-zh-sample-50.json'
    eval_path = r'D:\DissertationProject\DissertationProject\eval/eval_traj/BrowseComp-ZH_traj/SFT_IterScholar/20260406_220115_BrowseComp-ZH_traj_eval_results.json'
    output_path = r'/eval/BrowseComp-ZH/prediction_results/formatted_eval_results.json'

    # 检查文件是否存在
    if not os.path.exists(source_path):
        print(f"错误：找不到源文件 {source_path}")
        return
    if not os.path.exists(eval_path):
        print(f"错误：找不到评估结果文件 {eval_path}")
        return

    # 2. 读取原始数据集并建立映射：{question: id}
    print("正在建立问题-ID映射表...")
    try:
        with open(source_path, 'r', encoding='utf-8') as f:
            source_data = json.load(f)
        
        # 使用 strip() 去除可能存在的首尾空格，确保匹配准确
        q_to_id = {item['question'].strip(): item['id'] for item in source_data}
    except Exception as e:
        print(f"读取源文件失败: {e}")
        return

    # 3. 读取评估结果并进行格式转换
    print("正在转换格式...")
    try:
        with open(eval_path, 'r', encoding='utf-8') as f:
            eval_results = json.load(f)
        
        formatted_output = {}
        missing_count = 0

        for entry in eval_results:
            # 这里的 question 字段可能在评估结果的字典里
            q_text = entry.get('question', '').strip()
            
            if q_text in q_to_id:
                task_id = q_to_id[q_text]
                # 以 id 为 key，存储该条评估的所有信息
                formatted_output[task_id] = entry
            else:
                missing_count += 1
                # 如果匹配不到，可能是因为文本存在微小差异
                # print(f"警告：未找到匹配 ID 的问题: {q_text[:30]}...")

        # 4. 保存转换后的结果
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(formatted_output, f, ensure_ascii=False, indent=4)
        
        print(f"\n处理完成！")
        print(f"- 成功匹配并转换: {len(formatted_output)} 条")
        print(f"- 未能匹配条数: {missing_count}")
        print(f"- 结果已保存至: {output_path}")

    except Exception as e:
        print(f"处理评估结果失败: {e}")

if __name__ == "__main__":
    format_eval_results()

正在建立问题-ID映射表...
正在转换格式...

处理完成！
- 成功匹配并转换: 50 条
- 未能匹配条数: 0
- 结果已保存至: D:\DissertationProject\DissertationProject\eval/eval_traj/BrowseComp-ZH_traj/SFT_IterScholar/formatted_eval_results.json


In [11]:
from openai import OpenAI
import os

from dotenv import load_dotenv
load_dotenv()

# 这里的 API 地址直接写 localhost 或 127.0.0.1
client = OpenAI(
    base_url=os.getenv("DMX_API_BASE"),
    api_key=os.getenv("DMX_API_KEY"),
)

# 替换成你实际部署的模型名字（可通过上面的 curl 命令查看）

response = client.chat.completions.create(
    model="qwen3.5-flash",
    messages=[{"role": "user", "content": "你好，请确认你的身份。"}]
)

print(f"模型回复: {response.choices[0].message.content}")

PermissionDeniedError: Error code: 403 - {'error': {'message': '用户额度不足, 剩余额度: ＄-0.209618 (request id: 20260407142508816889045dUzZPrlo)', 'type': 'rix_api_error', 'param': '', 'code': 'insufficient_user_quota'}}